# 05. EXAONE-Deep-7.8B LoRA v2 Evaluation (Colab A100)

**Purpose**: LoRA v2 재학습 모델의 정량 평가 및 v1 대비 성능 비교

**Model Configuration**:
- **Base model**: `LGAI-EXAONE/EXAONE-Deep-7.8B` (revision: `17b70148e344`)
- **LoRA adapter**: `umyunsang/GovOn-EXAONE-LoRA-v2`
- **Test data**: `data/processed/v2_test.jsonl` (1,265 samples, 8 categories)

**Key Design Decisions**:
1. **transformers 4.44~4.49**: LoRA 학습 시점과 동일한 버전 사용
2. **EXAONE revision 고정** (`17b70148e344`): 학습 호환 코드 사용
3. **pad_token = unk_token**: EOS 학습을 위해 pad/eos 분리
4. **max_new_tokens=512**: v2 참조 답변 길이 증가 반영

**Metrics**: SacreBLEU, ROUGE-L (F1), BERTScore (F1), EOS Rate, Latency

---

## 1. Setup & Configuration

In [ ]:
# === 1-A. 패키지 설치 ===
# LoRA 학습 시점(2025-03)에는 transformers 4.44~4.49 + EXAONE 초기 코드 사용
# -> 최신 transformers는 EXAONE 최신 modeling 코드와만 호환되므로 4.44로 고정

!pip uninstall -y transformers accelerate -q
!pip install -q "transformers>=4.44,<4.50" "accelerate>=1.3.0,<2.0" peft
!pip install -q "numpy>=1.26,<3" "pandas>=2.2.2" sacrebleu rouge_score bert_score wandb tqdm bitsandbytes tabulate "protobuf<5.0.0"

# 설치 확인
import transformers, peft
v = tuple(int(x) for x in transformers.__version__.split('.')[:2])
assert v >= (4, 44) and v < (4, 50), \
    f"transformers >=4.44,<4.50 필요! 현재: {transformers.__version__}"
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")

print("\n설치 완료! [런타임] -> [세션 다시 시작] 후 아래 셀부터 진행하세요.")

In [ ]:
# === 1-B. API 키/토큰 설정 ===
# 방법 1: Colab Secrets (권장)
# 방법 2: 아래 주석 해제 후 직접 입력
import os

# --- HuggingFace Token ---
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("HF_TOKEN: Colab Secrets에서 로드됨")
except (ImportError, Exception):
    HF_TOKEN = None
    print("HF_TOKEN: Colab Secrets 사용 불가")

if not HF_TOKEN:
    # HF_TOKEN = "hf_your_token_here"  # 직접 입력 시 주석 해제
    pass

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# --- W&B API Key ---
try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    print("WANDB_API_KEY: Colab Secrets에서 로드됨")
except (ImportError, Exception):
    WANDB_API_KEY = None
    print("WANDB_API_KEY: Colab Secrets 사용 불가")

if not WANDB_API_KEY:
    # WANDB_API_KEY = "wandb_v1_your_key_here"  # 직접 입력 시 주석 해제
    pass

if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY

print("\nAPI 키 설정 완료.")

In [ ]:
# === 1-C. 환경 초기화 ===
import os, sys, torch, transformers, wandb

print(f'transformers {transformers.__version__}')
print(f'torch {torch.__version__}')

# EXAONE 모델의 학습 시점 revision 고정
EXAONE_REVISION = "17b70148e344"

# 프로젝트 클론
if not os.path.exists('/content/GovOn'):
    !git clone https://github.com/GovOn-Org/GovOn.git /content/GovOn
os.chdir('/content/GovOn')
PROJECT_ROOT = '/content/GovOn'
TEST_PATH = f'{PROJECT_ROOT}/data/processed/v2_test.jsonl'

# GPU 확인
if torch.cuda.is_available():
    print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# W&B 로그인
if os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
    print('W&B logged in.')
else:
    print('WARNING: WANDB_API_KEY가 설정되지 않았습니다.')

print('\n환경 초기화 완료.')

In [ ]:
# === 1-D. Configuration ===
import torch
from dataclasses import dataclass, field
from typing import Optional, List, Dict


@dataclass
class EvalConfig:
    """Central configuration for the v2 evaluation pipeline."""

    # --- Model ---
    base_model_id: str = "LGAI-EXAONE/EXAONE-Deep-7.8B"
    lora_adapter_id: str = "umyunsang/GovOn-EXAONE-LoRA-v2"

    # --- Data ---
    test_data_path: str = ""

    # --- Generation parameters ---
    max_new_tokens: int = 512
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.1
    do_sample: bool = False

    # --- System message ---
    system_message: str = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."

    # --- BERTScore model ---
    bertscore_model: str = "bert-base-multilingual-cased"

    # --- W&B ---
    wandb_project: str = "GovOn-retrain-v2"
    wandb_entity: Optional[str] = None
    wandb_run_name: Optional[str] = None
    wandb_tags: List[str] = field(default_factory=lambda: ["evaluation", "lora-v2"])

    # --- Output ---
    output_dir: str = ""

    def __post_init__(self):
        if not self.test_data_path:
            self.test_data_path = TEST_PATH
        if not self.output_dir:
            self.output_dir = f"{PROJECT_ROOT}/docs/outputs/experiments"
        os.makedirs(self.output_dir, exist_ok=True)


# v1 평가 결과 (비교용 하드코딩)
V1_RESULTS = {
    "sacrebleu": 0.5347,
    "rouge_l_f1": 4.1974,
    "bertscore_f1": 59.1461,
    "avg_latency": 27.023,
    "eos_rate": 0.0,
    "avg_gen_length": 212.73,
    "avg_ref_length": 336.11,
    "length_ratio": 0.6329,
}

config = EvalConfig()

print(f"Base model      : {config.base_model_id}")
print(f"LoRA adapter    : {config.lora_adapter_id}")
print(f"Max new tokens  : {config.max_new_tokens}")
print(f"Test data       : {config.test_data_path}")
print(f"System message  : {config.system_message}")
print(f"W&B project     : {config.wandb_project}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Test Data Loading & Preprocessing

In [ ]:
# === 2-A. Text Preprocessing ===
import re

# PII token patterns
_PII_PATTERNS = [
    r'▲+',
    r'\[NAME_MASKED\]',
    r'\[이름\]',
    r'\[전화번호\]',
    r'\[주소\]',
    r'<NAME>',
    r'<MOBILE_NUMBER>',
    r'<PHONE_NUMBER>',
    r'<DATE>',
    r'<BIRTH_NUMBER>',
    r'<ACCOUNT_NUMBER>',
    r'<ADDRESS>',
]
_PII_REGEX = re.compile('|'.join(_PII_PATTERNS))
_THOUGHT_REGEX = re.compile(r'<thought>.*?</thought>', flags=re.DOTALL)


def remove_thought_tags(text: str) -> str:
    """Remove <thought>...</thought> blocks."""
    return _THOUGHT_REGEX.sub('', text)


def remove_pii_tokens(text: str) -> str:
    """Remove PII placeholder tokens."""
    return _PII_REGEX.sub('', text)


def normalize_light(text: str) -> str:
    """경량 정규화: SacreBLEU용 (문장부호 유지).
    1. Remove <thought> tags
    2. Remove PII tokens
    3. Collapse whitespace
    """
    text = remove_thought_tags(text)
    text = remove_pii_tokens(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_text(text: str) -> str:
    """Full normalization for ROUGE-L / BERTScore.
    1. Remove <thought> tags
    2. Remove PII tokens
    3. Remove punctuation
    4. Collapse whitespace
    """
    text = remove_thought_tags(text)
    text = remove_pii_tokens(text)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# 검증
sample_raw = """<thought>
1. Complaint Type Analysis: blah blah
</thought>
안녕하십니까. [NAME_MASKED] 님, <NAME> 담당자입니다. <DATE>에 접수된 건입니다."""

print(f"Raw:              {sample_raw[:80]}...")
print(f"Light (for BLEU): {normalize_light(sample_raw)}")
print(f"Full  (for ROUGE): {normalize_text(sample_raw)}")

In [ ]:
# === 2-B. Load v2 Test Data ===
import json
import pandas as pd
from collections import Counter


def parse_v2_text(text: str) -> dict:
    """v2 JSONL의 text 필드에서 system/user/assistant 파싱.
    Format: [|system|]...[|endofturn|]\n[|user|]...[|endofturn|]\n[|assistant|]...[|endofturn|]
    """
    # assistant 파트 추출 (참조 답변)
    parts = text.split('[|assistant|]')
    if len(parts) >= 2:
        reference = parts[1].replace('[|endofturn|]', '').strip()
    else:
        reference = ""

    # user 파트 추출 (입력 프롬프트)
    user_match = re.search(r'\[\|user\|\](.*?)\[\|endofturn\|\]', text, re.DOTALL)
    user_content = user_match.group(1).strip() if user_match else ""

    # 프롬프트 = system + user까지 (assistant 생성 프롬프트)
    prompt = parts[0] + '[|assistant|]'

    return {
        "prompt": prompt,
        "user_content": user_content,
        "reference": reference,
    }


def load_v2_test_data(path: str) -> List[Dict]:
    """Load v2 JSONL test set.
    Each line has: text, category, id
    """
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"WARNING: Skipping line {line_no} - JSON decode error: {e}")
                continue

            parsed = parse_v2_text(item['text'])
            data.append({
                'id': item.get('id', f'sample_{line_no}'),
                'category': item.get('category', 'unknown'),
                'prompt': parsed['prompt'],
                'user_content': parsed['user_content'],
                'reference': parsed['reference'],
                'text': item['text'],
            })
    return data


test_data = load_v2_test_data(config.test_data_path)
print(f"Loaded {len(test_data)} test samples from {config.test_data_path}")

# 카테고리 분포
cat_counts = Counter(item['category'] for item in test_data)
print(f"\nCategory distribution ({len(cat_counts)} categories):")
for cat, count in cat_counts.most_common():
    print(f"  {cat:12s} : {count:5d} ({count / len(test_data) * 100:.1f}%)")

# 참조 답변 길이 통계
ref_lengths = [len(item['reference']) for item in test_data]
print(f"\nReference output length (chars):")
print(f"  mean={sum(ref_lengths)/len(ref_lengths):.0f}, "
      f"min={min(ref_lengths)}, max={max(ref_lengths)}, "
      f"median={sorted(ref_lengths)[len(ref_lengths)//2]}")

# 샘플 확인
print(f"\n--- Sample 0 ---")
print(f"  ID       : {test_data[0]['id']}")
print(f"  Category : {test_data[0]['category']}")
print(f"  Prompt   : {test_data[0]['prompt'][:150]}...")
print(f"  Reference: {test_data[0]['reference'][:150]}...")

## 3. Model Loading

In [ ]:
# === 3-A. QLoRA 4bit + LoRA v2 모델 로드 ===
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"  Base model : {config.base_model_id}")
print(f"  Revision   : {EXAONE_REVISION}")
print(f"  LoRA adapter: {config.lora_adapter_id}")

# 1. 토크나이저 (학습 시점 revision)
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_id,
    trust_remote_code=True,
    revision=EXAONE_REVISION,
)
# v2 핵심: pad_token = unk_token (EOS 학습을 위해 pad/eos 분리)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"

print(f"  eos_token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"  pad_token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"  unk_token: '{tokenizer.unk_token}' (id={tokenizer.unk_token_id})")

# 2. Base 모델 (4-bit 양자화, 학습 시점 revision)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_id,
    revision=EXAONE_REVISION,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# 3. LoRA v2 어댑터 적용
model = PeftModel.from_pretrained(base_model, config.lora_adapter_id)
model.eval()

print(f"\n  vocab_size: {len(tokenizer)}")
print(f"  pad != eos: {tokenizer.pad_token_id != tokenizer.eos_token_id}")
print("Model (base + LoRA v2) loaded.")

In [ ]:
# === 3-B. Sanity Check: 1개 샘플로 모델 동작 검증 ===
import time

sample = test_data[0]
prompt = sample['prompt']
print(f"프롬프트 앞부분:\n{prompt[:300]}...\n")

# EOS 토큰 ID 확인 (EXAONE: eos_token + endofturn 토큰)
eos_ids = [tokenizer.eos_token_id]
for special_tok in ['[|endofturn|]', '<|endofturn|>']:
    tok_id = tokenizer.convert_tokens_to_ids(special_tok)
    if tok_id is not None and tok_id != tokenizer.unk_token_id and tok_id not in eos_ids:
        eos_ids.append(tok_id)
print(f"EOS token IDs: {eos_ids}")
print(f"  eos_token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
input_len = inputs.input_ids.shape[1]
print(f"입력 토큰 수: {input_len}")

start = time.perf_counter()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        do_sample=False,
        repetition_penalty=config.repetition_penalty,
        eos_token_id=eos_ids,
    )
elapsed = time.perf_counter() - start

generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
gen_tokens = outputs[0].shape[0] - input_len

# EOS 생성 여부 확인
last_token_id = outputs[0][-1].item()
generated_eos = last_token_id in eos_ids

print(f"생성 토큰 수: {gen_tokens}")
print(f"소요 시간: {elapsed:.1f}s ({gen_tokens/elapsed:.1f} tok/s)")
print(f"EOS 생성: {generated_eos} (last_token_id={last_token_id})")
print(f"전체 데이터 예상 시간: {elapsed * len(test_data) / 3600:.1f}h")
print(f"\n--- 생성 결과 (첫 500자) ---")
print(generated[:500])
print(f"\n--- 참조 답변 (첫 300자) ---")
print(sample['reference'][:300])

if gen_tokens >= config.max_new_tokens - 1:
    print("\n[경고] max_new_tokens까지 생성됨 - EOS 미인식 가능성")
else:
    print(f"\n[OK] 정상 종료 (EOS 인식, {gen_tokens} tokens)")

## 4. Generation Functions

In [ ]:
# === 4-A. Generation Functions ===
import time
import traceback
from tqdm.auto import tqdm


def get_eos_ids():
    """EXAONE의 모든 EOS 토큰 ID를 반환"""
    ids = [tokenizer.eos_token_id]
    for tok in ['[|endofturn|]', '<|endofturn|>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id and tid not in ids:
            ids.append(tid)
    return ids


def generate_single(prompt: str, cfg: EvalConfig) -> dict:
    """단일 샘플 생성. 생성 텍스트, latency, EOS 생성 여부를 반환."""
    if not prompt:
        return {"text": "", "latency": 0.0, "eos_generated": False, "gen_tokens": 0}

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=4096
    ).to(model.device)
    input_len = inputs.input_ids.shape[1]

    eos_id_list = get_eos_ids()

    start = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=cfg.do_sample,
            repetition_penalty=cfg.repetition_penalty,
            eos_token_id=eos_id_list,
        )
    latency = time.perf_counter() - start

    gen_token_ids = outputs[0][input_len:]
    gen_tokens = len(gen_token_ids)
    generated_text = tokenizer.decode(gen_token_ids, skip_special_tokens=True)

    # EOS 생성 여부: 마지막 토큰이 EOS ID 중 하나인지 확인
    last_token_id = gen_token_ids[-1].item() if gen_tokens > 0 else -1
    eos_generated = last_token_id in eos_id_list

    return {
        "text": generated_text,
        "latency": latency,
        "eos_generated": eos_generated,
        "gen_tokens": gen_tokens,
    }


def generate_batch(items: List[Dict], cfg: EvalConfig) -> tuple:
    """전체 테스트 셋 생성. (generations, latencies, eos_flags) 반환."""
    generations = []
    latencies = []
    eos_flags = []

    for item in tqdm(items, desc="Generating"):
        try:
            result = generate_single(item['prompt'], cfg)
            generations.append(result['text'])
            latencies.append(result['latency'])
            eos_flags.append(result['eos_generated'])
        except Exception as e:
            print(f"ERROR on {item.get('id', '?')}: {e}")
            traceback.print_exc()
            generations.append("")
            latencies.append(0.0)
            eos_flags.append(False)

    return generations, latencies, eos_flags


print("Generation functions ready.")

## 5. Metric Computation

In [ ]:
# === 5-A. Metric Functions ===
import sacrebleu
from rouge_score import rouge_scorer
import bert_score
import numpy as np


def compute_sacrebleu(hypotheses: List[str], references: List[str]) -> dict:
    """Compute corpus-level SacreBLEU (intl tokenizer for Korean)."""
    bleu = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='intl')
    return {
        "sacrebleu_score": bleu.score,
        "sacrebleu_bp": bleu.bp,
        "sacrebleu_precisions": bleu.precisions,
        "sacrebleu_signature": str(bleu),
    }


def compute_rouge_l(hypotheses: List[str], references: List[str]) -> dict:
    """Compute ROUGE-L F-measure per sample and corpus average."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    scores = []
    for ref, hyp in zip(references, hypotheses):
        result = scorer.score(ref, hyp)
        scores.append(result["rougeL"].fmeasure)
    return {
        "rouge_l_f1_mean": np.mean(scores) * 100,
        "rouge_l_f1_std": np.std(scores) * 100,
        "rouge_l_f1_per_sample": [s * 100 for s in scores],
    }


def compute_bertscore(hypotheses: List[str], references: List[str],
                      model_type: str = "bert-base-multilingual-cased") -> dict:
    """Compute BERTScore F1 using multilingual BERT."""
    P, R, F1 = bert_score.score(
        hypotheses, references,
        model_type=model_type,
        lang="ko",
        verbose=True,
    )
    return {
        "bertscore_f1_mean": F1.mean().item() * 100,
        "bertscore_f1_std": F1.std().item() * 100,
        "bertscore_precision_mean": P.mean().item() * 100,
        "bertscore_recall_mean": R.mean().item() * 100,
        "bertscore_f1_per_sample": (F1 * 100).tolist(),
    }


def compute_all_metrics(hypotheses: List[str], references: List[str],
                         latencies: List[float], eos_flags: List[bool],
                         cfg: EvalConfig) -> dict:
    """Compute all metrics with appropriate normalization per metric type."""

    # SacreBLEU: 경량 정규화 (문장부호 유지)
    light_hyps = [normalize_light(h) or "(empty)" for h in hypotheses]
    light_refs = [normalize_light(r) or "(empty)" for r in references]

    # ROUGE-L, BERTScore: 풀 정규화
    norm_hyps = [normalize_text(h) or "(empty)" for h in hypotheses]
    norm_refs = [normalize_text(r) or "(empty)" for r in references]

    print("Computing SacreBLEU (light normalization, intl tokenizer)...")
    bleu_metrics = compute_sacrebleu(light_hyps, light_refs)

    print("Computing ROUGE-L (full normalization)...")
    rouge_metrics = compute_rouge_l(norm_hyps, norm_refs)

    print("Computing BERTScore (full normalization)...")
    bert_metrics = compute_bertscore(norm_hyps, norm_refs, model_type=cfg.bertscore_model)

    # EOS 생성률 (v2 핵심 지표)
    eos_count = sum(eos_flags)
    eos_rate = eos_count / len(eos_flags) * 100 if eos_flags else 0.0

    # Latency metrics
    valid_latencies = [l for l in latencies if l > 0]
    latency_metrics = {
        "avg_latency_sec": np.mean(valid_latencies) if valid_latencies else 0,
        "p50_latency_sec": np.percentile(valid_latencies, 50) if valid_latencies else 0,
        "p95_latency_sec": np.percentile(valid_latencies, 95) if valid_latencies else 0,
    }

    # Generation length stats
    gen_lengths = [len(h) for h in light_hyps]
    ref_lengths = [len(r) for r in light_refs]
    length_metrics = {
        "avg_gen_length_chars": np.mean(gen_lengths),
        "avg_ref_length_chars": np.mean(ref_lengths),
        "length_ratio": np.mean(gen_lengths) / max(np.mean(ref_lengths), 1),
    }

    return {
        **bleu_metrics,
        **rouge_metrics,
        **bert_metrics,
        "eos_rate": eos_rate,
        "eos_count": eos_count,
        "eos_total": len(eos_flags),
        **latency_metrics,
        **length_metrics,
    }


print("Metric computation functions defined.")

## 6. Run Full Evaluation

In [ ]:
# === 6-A. Generate predictions ===
print(f"Generating predictions for {len(test_data)} samples...")
print(f"  max_new_tokens     = {config.max_new_tokens}")
print(f"  temperature        = {config.temperature}")
print(f"  repetition_penalty = {config.repetition_penalty}")
print(f"  do_sample          = {config.do_sample}")
print()

generations, latencies, eos_flags = generate_batch(test_data, config)

references = [item['reference'] for item in test_data]

print(f"\nGeneration complete. {len(generations)} outputs produced.")
print(f"Total time: {sum(latencies):.1f}s ({sum(latencies)/3600:.2f}h)")
print(f"EOS generated: {sum(eos_flags)}/{len(eos_flags)} ({sum(eos_flags)/len(eos_flags)*100:.1f}%)")

In [ ]:
# === 6-B. Compute all metrics (corpus-level) ===
corpus_metrics = compute_all_metrics(generations, references, latencies, eos_flags, config)

print("\n" + "=" * 60)
print("CORPUS-LEVEL RESULTS (LoRA v2)")
print("=" * 60)
print(f"  SacreBLEU          : {corpus_metrics['sacrebleu_score']:.4f}")
print(f"  BLEU Brevity Pen.  : {corpus_metrics['sacrebleu_bp']:.4f}")
print(f"  ROUGE-L F1         : {corpus_metrics['rouge_l_f1_mean']:.4f} (+/- {corpus_metrics['rouge_l_f1_std']:.2f})")
print(f"  BERTScore F1       : {corpus_metrics['bertscore_f1_mean']:.4f} (+/- {corpus_metrics['bertscore_f1_std']:.2f})")
print(f"  BERTScore P / R    : {corpus_metrics['bertscore_precision_mean']:.4f} / {corpus_metrics['bertscore_recall_mean']:.4f}")
print(f"  EOS Rate           : {corpus_metrics['eos_rate']:.1f}% ({corpus_metrics['eos_count']}/{corpus_metrics['eos_total']})")
print(f"  Avg Latency        : {corpus_metrics['avg_latency_sec']:.3f}s")
print(f"  p50 Latency        : {corpus_metrics['p50_latency_sec']:.3f}s")
print(f"  p95 Latency        : {corpus_metrics['p95_latency_sec']:.3f}s")
print(f"  Avg Gen Length     : {corpus_metrics['avg_gen_length_chars']:.0f} chars")
print(f"  Avg Ref Length     : {corpus_metrics['avg_ref_length_chars']:.0f} chars")
print(f"  Length Ratio       : {corpus_metrics['length_ratio']:.4f}")
print("=" * 60)

## 7. v1 vs v2 Comparison

In [ ]:
# === 7-A. v1 vs v2 비교 테이블 ===
from tabulate import tabulate

v2_results = {
    "sacrebleu": corpus_metrics['sacrebleu_score'],
    "rouge_l_f1": corpus_metrics['rouge_l_f1_mean'],
    "bertscore_f1": corpus_metrics['bertscore_f1_mean'],
    "avg_latency": corpus_metrics['avg_latency_sec'],
    "eos_rate": corpus_metrics['eos_rate'],
    "avg_gen_length": corpus_metrics['avg_gen_length_chars'],
    "avg_ref_length": corpus_metrics['avg_ref_length_chars'],
    "length_ratio": corpus_metrics['length_ratio'],
}

comparison_rows = []
for metric in ["sacrebleu", "rouge_l_f1", "bertscore_f1", "avg_latency",
               "eos_rate", "avg_gen_length", "avg_ref_length", "length_ratio"]:
    v1_val = V1_RESULTS[metric]
    v2_val = v2_results[metric]
    diff = v2_val - v1_val

    # 방향성 결정 (낮을수록 좋은 지표)
    lower_is_better = metric in ["avg_latency"]
    if lower_is_better:
        direction = "+" if diff < 0 else ("-" if diff > 0 else "=")
    else:
        direction = "+" if diff > 0 else ("-" if diff < 0 else "=")

    comparison_rows.append([
        metric,
        f"{v1_val:.4f}",
        f"{v2_val:.4f}",
        f"{diff:+.4f}",
        direction,
    ])

print("\n" + "=" * 75)
print("v1 vs v2 COMPARISON")
print("=" * 75)
print(tabulate(
    comparison_rows,
    headers=["Metric", "v1", "v2", "Delta", "Dir"],
    tablefmt="grid",
    stralign="right",
))
print("=" * 75)

# 핵심 개선 포인트 요약
print("\n[핵심 비교]")
print(f"  EOS Rate    : {V1_RESULTS['eos_rate']:.1f}% -> {v2_results['eos_rate']:.1f}% (v2 핵심 개선 목표)")
print(f"  SacreBLEU   : {V1_RESULTS['sacrebleu']:.4f} -> {v2_results['sacrebleu']:.4f}")
print(f"  ROUGE-L F1  : {V1_RESULTS['rouge_l_f1']:.4f} -> {v2_results['rouge_l_f1']:.4f}")
print(f"  BERTScore F1: {V1_RESULTS['bertscore_f1']:.4f} -> {v2_results['bertscore_f1']:.4f}")
print(f"  Length Ratio: {V1_RESULTS['length_ratio']:.4f} -> {v2_results['length_ratio']:.4f}")

## 8. Category-Level Evaluation

In [ ]:
# === 8-A. 카테고리별 성능 분석 ===
from collections import defaultdict

CATEGORIES = ["교통", "환경", "건축", "복지", "세금", "안전", "행정", "기타"]


def compute_category_metrics(test_data: List[Dict],
                              generations: List[str],
                              references: List[str],
                              latencies: List[float],
                              eos_flags: List[bool],
                              cfg: EvalConfig) -> pd.DataFrame:
    """Compute metrics grouped by category."""
    cat_indices = defaultdict(list)
    for i, item in enumerate(test_data):
        cat_indices[item['category']].append(i)

    rows = []
    for cat in CATEGORIES:
        indices = cat_indices.get(cat, [])
        if not indices:
            continue

        cat_gens = [generations[i] for i in indices]
        cat_refs = [references[i] for i in indices]
        cat_lats = [latencies[i] for i in indices]
        cat_eos = [eos_flags[i] for i in indices]

        # Normalize
        light_hyps = [normalize_light(h) or "(empty)" for h in cat_gens]
        light_refs = [normalize_light(r) or "(empty)" for r in cat_refs]
        norm_hyps = [normalize_text(h) or "(empty)" for h in cat_gens]
        norm_refs = [normalize_text(r) or "(empty)" for r in cat_refs]

        # SacreBLEU
        bleu = sacrebleu.corpus_bleu(light_hyps, [light_refs], tokenize='intl')

        # ROUGE-L
        r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
        rouge_scores = [r_scorer.score(r, h)["rougeL"].fmeasure for r, h in zip(norm_refs, norm_hyps)]

        # BERTScore
        if len(norm_hyps) >= 2:
            P, R, F1 = bert_score.score(
                norm_hyps, norm_refs,
                model_type=cfg.bertscore_model,
                lang="ko", verbose=False,
            )
            bs_f1 = F1.mean().item() * 100
        else:
            bs_f1 = float('nan')

        # EOS rate
        eos_rate = sum(cat_eos) / len(cat_eos) * 100

        rows.append({
            "category": cat,
            "n_samples": len(indices),
            "sacrebleu": bleu.score,
            "rouge_l_f1": np.mean(rouge_scores) * 100,
            "bertscore_f1": bs_f1,
            "eos_rate": eos_rate,
            "avg_latency": np.mean(cat_lats),
            "avg_gen_len": np.mean([len(h) for h in light_hyps]),
            "avg_ref_len": np.mean([len(r) for r in light_refs]),
        })

    # 미분류 카테고리 처리
    known_cats = set(CATEGORIES)
    other_indices = [i for i, item in enumerate(test_data) if item['category'] not in known_cats]
    if other_indices:
        print(f"WARNING: {len(other_indices)} samples with unknown categories")

    return pd.DataFrame(rows)


cat_df = compute_category_metrics(test_data, generations, references, latencies, eos_flags, config)

print("\n" + "=" * 110)
print("CATEGORY-LEVEL RESULTS")
print("=" * 110)
print(cat_df.to_string(index=False, float_format="{:.2f}".format))

# 최고/최저 카테고리
if len(cat_df) > 1:
    best_cat = cat_df.loc[cat_df['bertscore_f1'].idxmax()]
    worst_cat = cat_df.loc[cat_df['bertscore_f1'].idxmin()]
    print(f"\nBest category  (BERTScore): {best_cat['category']} ({best_cat['bertscore_f1']:.2f})")
    print(f"Worst category (BERTScore): {worst_cat['category']} ({worst_cat['bertscore_f1']:.2f})")
print("=" * 110)

## 9. W&B Logging

In [ ]:
# === 9-A. W&B Run 초기화 및 로깅 ===
from datetime import datetime

run_name = config.wandb_run_name or f"eval-lora-v2-{datetime.now().strftime('%m%d-%H%M')}"

run = wandb.init(
    project=config.wandb_project,
    entity=config.wandb_entity,
    name=run_name,
    tags=config.wandb_tags,
    config={
        "model": f"{config.base_model_id} + {config.lora_adapter_id}",
        "base_model": config.base_model_id,
        "lora_adapter": config.lora_adapter_id,
        "exaone_revision": EXAONE_REVISION,
        "max_new_tokens": config.max_new_tokens,
        "temperature": config.temperature,
        "repetition_penalty": config.repetition_penalty,
        "do_sample": config.do_sample,
        "system_message": config.system_message,
        "n_test_samples": len(test_data),
        "bertscore_model": config.bertscore_model,
        "normalization": "thought_tags + PII_tokens + whitespace",
        "pad_token_strategy": "unk_token (EOS 분리)",
    },
)

print(f"W&B run: {run.name} ({run.url})")

# --- Corpus-level metrics ---
scalar_metrics = {
    k: v for k, v in corpus_metrics.items()
    if not isinstance(v, (list, str))
}
wandb.log(scalar_metrics)
wandb.run.summary["sacrebleu_signature"] = corpus_metrics.get("sacrebleu_signature", "")

# --- v1 비교 결과 ---
for metric in V1_RESULTS:
    wandb.run.summary[f"v1/{metric}"] = V1_RESULTS[metric]
    if metric in v2_results:
        wandb.run.summary[f"delta/{metric}"] = v2_results[metric] - V1_RESULTS[metric]

print("Corpus-level metrics logged.")

In [ ]:
# === 9-B. Per-sample results W&B Table ===
columns = [
    "id", "category",
    "reference", "prediction",
    "ref_normalized", "pred_normalized",
    "rouge_l_f1", "bertscore_f1",
    "eos_generated",
    "ref_length", "pred_length", "latency_sec",
]

table_data = []
rouge_per_sample = corpus_metrics.get("rouge_l_f1_per_sample", [0] * len(test_data))
bert_per_sample = corpus_metrics.get("bertscore_f1_per_sample", [0] * len(test_data))

for i, item in enumerate(test_data):
    norm_ref = normalize_text(references[i])
    norm_pred = normalize_text(generations[i])
    table_data.append([
        item.get("id", f"sample_{i}"),
        item.get("category", "unknown"),
        references[i][:500],
        generations[i][:500],
        norm_ref[:500],
        norm_pred[:500],
        rouge_per_sample[i] if i < len(rouge_per_sample) else 0,
        bert_per_sample[i] if i < len(bert_per_sample) else 0,
        eos_flags[i],
        len(norm_ref),
        len(norm_pred),
        latencies[i],
    ])

results_table = wandb.Table(columns=columns, data=table_data)
wandb.log({"per_sample_results": results_table})

# Category table
cat_table = wandb.Table(dataframe=cat_df)
wandb.log({"category_metrics": cat_table})

# Category scalars
for _, row in cat_df.iterrows():
    cat_name = row["category"]
    wandb.run.summary[f"cat/{cat_name}/sacrebleu"] = row["sacrebleu"]
    wandb.run.summary[f"cat/{cat_name}/rouge_l_f1"] = row["rouge_l_f1"]
    wandb.run.summary[f"cat/{cat_name}/bertscore_f1"] = row["bertscore_f1"]
    wandb.run.summary[f"cat/{cat_name}/eos_rate"] = row["eos_rate"]
    wandb.run.summary[f"cat/{cat_name}/n_samples"] = row["n_samples"]

print(f"Per-sample table ({len(table_data)} rows) + category table logged.")

wandb.finish()
print("W&B run finished.")

## 10. Results Export & Report

In [ ]:
# === 10-A. Save results JSON ===
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result_filename = f"eval_lora_v2_{timestamp}.json"
result_path = os.path.join(config.output_dir, result_filename)

export_data = {
    "metadata": {
        "timestamp": timestamp,
        "model": f"{config.base_model_id} + {config.lora_adapter_id}",
        "base_model": config.base_model_id,
        "lora_adapter": config.lora_adapter_id,
        "exaone_revision": EXAONE_REVISION,
        "n_test_samples": len(test_data),
        "test_data_path": "data/processed/v2_test.jsonl",
        "system_message": config.system_message,
        "normalization": "thought_tags + PII_tokens + whitespace",
        "pad_token_strategy": "unk_token (EOS 분리)",
    },
    "generation_config": {
        "max_new_tokens": config.max_new_tokens,
        "temperature": config.temperature,
        "repetition_penalty": config.repetition_penalty,
        "do_sample": config.do_sample,
    },
    "corpus_metrics": {
        k: v for k, v in corpus_metrics.items()
        if not isinstance(v, list)
    },
    "v1_results": V1_RESULTS,
    "v2_results": v2_results,
    "v1_v2_delta": {
        k: v2_results[k] - V1_RESULTS[k]
        for k in V1_RESULTS if k in v2_results
    },
    "category_metrics": cat_df.to_dict(orient="records"),
    "per_sample_predictions": [
        {
            "id": test_data[i].get("id", f"sample_{i}"),
            "category": test_data[i].get("category", "unknown"),
            "reference": references[i],
            "prediction": generations[i],
            "eos_generated": eos_flags[i],
            "latency_sec": latencies[i],
        }
        for i in range(len(test_data))
    ],
}

with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2, default=str)

print(f"Results saved to: {result_path}")
print(f"File size: {os.path.getsize(result_path) / 1024 / 1024:.1f} MB")

In [ ]:
# === 10-B. Auto-download (Colab) ===
try:
    from google.colab import files
    files.download(result_path)
    print(f"다운로드 시작: {result_path}")
except ImportError:
    print("Colab 환경이 아닙니다. 수동으로 파일을 복사하세요.")

In [ ]:
# === 10-C. Final Summary Report ===

print("\n" + "=" * 75)
print("FINAL EVALUATION REPORT: EXAONE-Deep-7.8B LoRA v2")
print("=" * 75)

# v1 vs v2 비교
print("\n[v1 vs v2 Comparison]")
print(f"  {'Metric':<20} {'v1':>12} {'v2':>12} {'Delta':>12}")
print(f"  {'-'*20} {'-'*12} {'-'*12} {'-'*12}")
for metric in ["sacrebleu", "rouge_l_f1", "bertscore_f1", "eos_rate",
               "avg_latency", "avg_gen_length", "length_ratio"]:
    v1 = V1_RESULTS[metric]
    v2 = v2_results[metric]
    delta = v2 - v1
    print(f"  {metric:<20} {v1:>12.4f} {v2:>12.4f} {delta:>+12.4f}")

# EOS Rate 강조
print(f"\n[핵심 지표: EOS 생성률]")
print(f"  v1: {V1_RESULTS['eos_rate']:.1f}% -> v2: {v2_results['eos_rate']:.1f}%")
if v2_results['eos_rate'] > V1_RESULTS['eos_rate']:
    print(f"  EOS 학습 개선 효과 확인!")
else:
    print(f"  EOS 학습 효과 추가 분석 필요")

# 카테고리별 요약
print(f"\n[Category EOS Rate]")
for _, row in cat_df.iterrows():
    print(f"  {row['category']:8s}: EOS {row['eos_rate']:5.1f}% | "
          f"BLEU {row['sacrebleu']:6.2f} | "
          f"ROUGE-L {row['rouge_l_f1']:6.2f} | "
          f"BERTScore {row['bertscore_f1']:6.2f}")

# 샘플 예시
n_show = min(3, len(test_data))
print(f"\n[Sample Predictions ({n_show} examples)]")
for i in range(n_show):
    item = test_data[i]
    norm_ref = normalize_text(references[i])
    norm_gen = normalize_text(generations[i])
    rouge_i = corpus_metrics["rouge_l_f1_per_sample"][i] if i < len(corpus_metrics.get("rouge_l_f1_per_sample", [])) else 0
    bert_i = corpus_metrics["bertscore_f1_per_sample"][i] if i < len(corpus_metrics.get("bertscore_f1_per_sample", [])) else 0

    print(f"\n  --- [{i}] {item.get('id', '?')} | {item.get('category', '?')} | EOS: {eos_flags[i]} ---")
    print(f"  ROUGE-L: {rouge_i:.1f} | BERTScore: {bert_i:.1f} | Latency: {latencies[i]:.1f}s")
    print(f"  REF: {norm_ref[:200]}{'...' if len(norm_ref) > 200 else ''}")
    print(f"  GEN: {norm_gen[:200]}{'...' if len(norm_gen) > 200 else ''}")

print(f"\n{'=' * 75}")
print(f"Results file: {result_path}")
print("Pipeline complete.")
print(f"{'=' * 75}")